In [ ]:
# text_detection_complete.py
# 包含全部12个地标：思源楼、图书馆、南门、天佑会堂、世纪钟、知行碑、
# 明湖碑、芳华园碑、逸夫楼、机械楼、科学会堂、迎客松

import os
import ssl
import cv2
import json
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm

# 修复SSL
ssl._create_default_https_context = ssl._create_unverified_context
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''

from paddleocr import PaddleOCR

# ========== 配置 ==========
DATA_PATH = r"E:\object_detection\data"
OUTPUT_PATH = r"E:\object_detection\detection_results"

# 全部12个地标缩写映射（完整版）
LANDMARK_MAP = {
    'sy': '思源楼',
    'tsg': '图书馆', 
    'nm': '南门',
    'ty': '天佑会堂',
    'sjz': '世纪钟',
    'zx': '知行碑',
    'mh': '明湖碑',
    'fhy': '芳华园碑',
    'yf': '逸夫楼',
    'jx': '机械楼',
    'kx': '科学会堂',      # 新增
    'yk': '迎客松'         # 新增
}

# 需要检测的目标文字（用于实验报告）
TARGET_TEXTS = ['北京交通大学', 'BJTJU', '明湖', '知行', '芳华园', '科学会堂', '迎客松']
# ==========================

# 检查路径
if not os.path.exists(DATA_PATH):
    print(f"错误：路径不存在 - {DATA_PATH}")
    exit(1)

# 获取所有图片
image_files = [f for f in os.listdir(DATA_PATH) 
               if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

print(f"找到 {len(image_files)} 张图片")
print("\n图片按地标分类：")
landmark_count = {}
for f in image_files:
    # 提取前缀（支持 sy-xxx 或 kx-xxx 格式）
    prefix = f.split('-')[0] if '-' in f else 'unknown'
    if prefix in LANDMARK_MAP:
        landmark_count[prefix] = landmark_count.get(prefix, 0) + 1
    else:
        print(f"  未知格式: {f}")

for code, name in LANDMARK_MAP.items():
    count = landmark_count.get(code, 0)
    print(f"  {name}({code}): {count}张")

# 初始化OCR
print("\n正在初始化PaddleOCR...")
ocr = PaddleOCR(
    use_angle_cls=False,
    lang='ch',
    use_gpu=False,
    show_log=False,
    det_db_thresh=0.3,
    det_db_box_thresh=0.5
)
print("初始化完成！")

# 加载中文字体
def get_font(size=16):
    font_paths = [
        "C:/Windows/Fonts/simhei.ttf",   # 黑体
        "C:/Windows/Fonts/msyh.ttc",     # 微软雅黑
        "C:/Windows/Fonts/simsun.ttc"    # 宋体
    ]
    for font_path in font_paths:
        if os.path.exists(font_path):
            return ImageFont.truetype(font_path, size)
    return ImageFont.load_default()

font = get_font(16)

# 创建输出目录
os.makedirs(OUTPUT_PATH, exist_ok=True)
vis_folder = os.path.join(OUTPUT_PATH, 'visualization')
json_folder = os.path.join(OUTPUT_PATH, 'json_results')
os.makedirs(vis_folder, exist_ok=True)
os.makedirs(json_folder, exist_ok=True)

# 存储结果
all_results = {}

# 处理图片
print("\n开始文字检测...")
for img_name in tqdm(image_files, desc="检测进度"):
    img_path = os.path.join(DATA_PATH, img_name)
    
    # 读取图片
    img = cv2.imread(img_path)
    if img is None:
        print(f"  无法读取: {img_name}")
        continue
    
    # 文字检测
    try:
        result = ocr.ocr(img_path, cls=False)
    except Exception as e:
        print(f"  检测失败 {img_name}: {e}")
        all_results[img_name] = []
        continue
    
    # 转换颜色空间
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil_img)
    
    detections = []
    if result and result[0]:
        for line in result[0]:
            bbox = np.array(line[0], dtype=np.int32)
            text = line[1][0]
            confidence = line[1][1]
            detections.append({
                'bbox': bbox.tolist(),
                'text': text,
                'confidence': confidence
            })
            
            # 绘制边界框（绿色）
            points = [(p[0], p[1]) for p in bbox]
            draw.polygon(points, outline=(0, 255, 0), width=2)
            
            # 绘制文字标签
            x, y = bbox[0]
            label = f"{text} ({confidence:.2f})"
            try:
                bbox_size = draw.textbbox((x, y-18), label, font=font)
                draw.rectangle(bbox_size, fill=(0, 255, 0))
                draw.text((x, y-18), label, fill=(0, 0, 0), font=font)
            except:
                draw.text((x, y-15), label, fill=(0, 255, 0))
    
    all_results[img_name] = detections
    
    # 保存可视化结果
    result_img = np.array(pil_img)
    result_img = cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR)
    save_path = os.path.join(vis_folder, img_name)
    cv2.imwrite(save_path, result_img)

# 保存汇总JSON
summary = {
    'total_images': len(all_results),
    'images_with_text': sum(1 for v in all_results.values() if v),
    'total_text_boxes': sum(len(v) for v in all_results.values()),
    'results': {k: [{'text': d['text'], 'confidence': d['confidence']} 
                    for d in v] for k, v in all_results.items()}
}

with open(os.path.join(OUTPUT_PATH, 'detection_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# ========== 按地标统计（12个地标全部分析）==========
landmark_stats = {}
target_hit_stats = {}

for img_name, detections in all_results.items():
    prefix = img_name.split('-')[0] if '-' in img_name else 'unknown'
    landmark_name = LANDMARK_MAP.get(prefix, prefix)
    
    if landmark_name not in landmark_stats:
        landmark_stats[landmark_name] = {
            'total': 0,
            'with_text': 0,
            'total_boxes': 0,
            'detected_texts': []
        }
        target_hit_stats[landmark_name] = {target: 0 for target in TARGET_TEXTS}
    
    landmark_stats[landmark_name]['total'] += 1
    
    if detections:
        landmark_stats[landmark_name]['with_text'] += 1
        landmark_stats[landmark_name]['total_boxes'] += len(detections)
        detected_texts_str = ' '.join([d['text'] for d in detections])
        landmark_stats[landmark_name]['detected_texts'].extend([d['text'] for d in detections])
        
        # 统计目标文字命中情况
        for target in TARGET_TEXTS:
            if target in detected_texts_str:
                target_hit_stats[landmark_name][target] += 1

# 打印统计结果
print("\n" + "=" * 80)
print("检测完成！统计结果：")
print(f"  总图片数: {summary['total_images']}")
print(f"  包含文字的图片数: {summary['images_with_text']}")
print(f"  检测到的文本框总数: {summary['total_text_boxes']}")
print(f"  检测率: {summary['images_with_text']/summary['total_images']:.2%}")
print("=" * 80)

print("\n按地标统计:")
print("-" * 80)
print(f"{'地标名称':<12} {'图片数':<6} {'检出数':<6} {'检出率':<8} {'文本框数':<8}")
print("-" * 80)
for name, stats in landmark_stats.items():
    rate = stats['with_text'] / stats['total'] if stats['total'] > 0 else 0
    print(f"{name:<12} {stats['total']:<6} {stats['with_text']:<6} {rate:<8.0%} {stats['total_boxes']:<8}")

print("\n目标文字检测统计:")
print("-" * 80)
print(f"{'地标名称':<12}", end='')
for target in TARGET_TEXTS:
    print(f"{target:<16}", end='')
print()
print("-" * 80)
for name in landmark_stats.keys():
    print(f"{name:<12}", end='')
    for target in TARGET_TEXTS:
        total = landmark_stats[name]['total']
        hit = target_hit_stats[name][target]
        print(f"{hit}/{total:<14}", end='')
    print()

# 保存详细报告
report = {
    'summary': summary,
    'landmark_stats': {name: {k: v for k, v in stats.items() if k != 'detected_texts'} 
                       for name, stats in landmark_stats.items()},
    'target_hit_stats': target_hit_stats
}

with open(os.path.join(OUTPUT_PATH, 'landmark_report.json'), 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 80)
print(f"详细报告已保存: {os.path.join(OUTPUT_PATH, 'landmark_report.json')}")
print(f"可视化结果保存在: {vis_folder}")
print("=" * 80)

找到 1494 张图片

图片按地标分类：
  未知格式: mh-00iii0wer.jpg
  未知格式: mh-03e1ebaa-d9ca-4cb3-b0ed-770bec526323.png
  未知格式: mh-0513a4fad192097469b24a235baf8312.jpg
  未知格式: mh-091fbnqpasdm.jpeg
  未知格式: mh-092r0924r2owof.jpg
  未知格式: mh-0b18e68c28c84bb1c8d2c75e43081deb.jpg
  未知格式: mh-0inhgdxgrdh.png
  未知格式: mh-0ure0320r2ur02qr.jpg
  未知格式: mh-12bkjnl31.jpg
  未知格式: mh-1746580766579.jpg
  未知格式: mh-1746580766618.jpg
  未知格式: mh-1746580766659.jpg
  未知格式: mh-18dbdk.png
  未知格式: mh-192cd206af38c624813a96d9d7399d2.jpg
  未知格式: mh-1ds8bxn7qd.jpg
  未知格式: mh-1oibj.jpg
  未知格式: mh-20160428143934097154.jpg
  未知格式: mh-20240508165927.png
  未知格式: mh-20240508170651.jpg
  未知格式: mh-20240515220849.jpg
  未知格式: mh-20240515220850.jpg
  未知格式: mh-202405152208502.jpg
  未知格式: mh-202505061032561.jpg
  未知格式: mh-20250506103257.jpg
  未知格式: mh-20250506194920.jpg
  未知格式: mh-20250506194923.jpg
  未知格式: mh-20250506194927.jpg
  未知格式: mh-202505071933001.jpg
  未知格式: mh-20250507193301.jpg
  未知格式: mh-20250507193302.jpg
  未知格式: mh-20260511140022_134_

检测进度:   5%|▌         | 82/1494 [00:41<11:59,  1.96it/s]


KeyboardInterrupt: 